<img src="images/logo.jpg" style="display: block; margin-left: auto; margin-right: auto;" alt="לוגו הקורס">

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="width: 70%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
            <strong>זו גרסת האופליין של המחברת.</strong><br>היא מיועדת למצב שבו הרשת בכיתה או השירות באינטרנט לא עובדים או חסומים: אם הבקשה לאינטרנט נכשלת, הקוד טוען תשובה שמורה מקובץ בתיקייה <code>resources</code> ומשתמש בה.<br>חשוב להבין: <strong>הטריק הזה הוא עקיפה כיתתית בלבד</strong>. הוא לא חובה בבקשות רשת, ולא משהו שעושים בקוד אמיתי - תוכנית אמיתית פשוט מטפלת בשגיאה (מדפיסה הודעה או מנסה שוב).<br>המחברת הרגילה (כשיש אינטרנט): <code>Summary.ipynb</code>
        </p>
    </div>
    <div style="display: flex; width: 20%; border-right: 0.1rem solid #A5A5A5; padding: 1rem 2rem;">
        <p style="text-align: center; direction: rtl; justify-content: center; align-items: center; clear: both;">
            <strong>גרסת אופליין</strong><br>
            לא קוד אמיתי!
        </p>
    </div>
</div>

# <p style="text-align: right; direction: rtl; float: right;">יום 4 - תרגולים וסיכום (גרסת אופליין)</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
בתרגולים של היום נתרגל שליחת בקשות, קריאת JSON, שמירת קבצים והורדת תמונות.<br>
התרגילים מסודרים מהקל אל הקשה. תרגיל 6 הוא אתגר - אפשר לעשות אותו בבית.<br>
זו גרסת האופליין: אם השירות לא זמין, חלק מהתרגילים משתמשים בנתונים שמורים מהתיקייה <code>resources</code>. זו עקיפה כיתתית בלבד ולא משהו שצריך לעשות בקוד אמיתי.
</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/tip.png" style="height: 50px !important;" alt="טיפ">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>טיפ</strong>:<br>
            הצעד הראשון בכל תרגיל הוא <strong>להדפיס את התשובה של ה-API ולקרוא אותה</strong>, ולמצוא בה איפה נמצא החלק שאתם צריכים (באיזה מפתח, ובאיזה מילון או רשימה). זה חלק מהתרגיל - לא ידע שהייתם אמורים להביא איתכם, ולא משהו שפספסתם בשיעור. לכל API יש פורמט משלו, וכולם מגלים אותו בדיוק כך.
        </p>
    </div>
</div>

## <p style="text-align: right; direction: rtl; float: right; clear: both;">הכנה: הריצו את התא הבא</p>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
התא הבא מכין את כל מה שצריך לתרגילים: הפונקציה <code>get_api_data</code> שכתבנו, טבלת הערים עם קווי הרוחב והאורך,
ומילון שמתרגם את קוד מזג האוויר של Open-Meteo לתיאור בעברית.<br>
בגרסת האופליין הוא גם טוען שני קבצי תשובות שמורות (<code>SAMPLE_WEATHER</code> ו-<code>SAMPLE_QUOTES</code>) מהתיקייה <code>resources</code>. התרגילים משתמשים בהם רק כשבקשה נכשלת, כדי שאפשר יהיה להמשיך בכיתה. <strong>זו עקיפה כיתתית בלבד</strong> - תוכנית אמיתית הייתה פשוט מטפלת בשגיאה (מדפיסה הודעה או מנסה שוב) ולא טוענת קובץ.<br>
<strong>הריצו אותו פעם אחת בתחילת התרגולים</strong> (ושוב אם אתם מפעילים מחדש את הקרנל).
</p>

In [ ]:
import requests
import json
import os
import csv
import time
from datetime import datetime


def get_api_data(url, params=None, timeout=10):
    """שולחת בקשת GET ומחזירה מילון, או None אם משהו השתבש."""
    try:
        response = requests.get(url, params=params, timeout=timeout)
        if response.status_code == 200:
            return response.json()
        print(f"❌ קוד שגיאה: {response.status_code}")
    except requests.exceptions.RequestException:
        print("❌ לא הצלחנו להתחבר לשירות")
    return None


OPEN_METEO_URL = 'https://api.open-meteo.com/v1/forecast'
QUOTE_URL = 'https://dummyjson.com/quotes/random'

# עיר -> (קו רוחב, קו אורך)
CITIES = {
    'תל אביב': (32.0853, 34.7818),
    'ירושלים': (31.7683, 35.2137),
    'חיפה': (32.7940, 34.9896),
    'באר שבע': (31.2530, 34.7915),
    'אילת': (29.5577, 34.9519),
}

# קוד מזג אוויר של Open-Meteo -> תיאור בעברית
WEATHER_CODES = {
    0: 'שמיים בהירים', 1: 'בהיר ברובו', 2: 'מעונן חלקית', 3: 'מעונן',
    45: 'ערפל', 48: 'ערפל', 51: 'טפטוף קל', 53: 'טפטוף', 55: 'טפטוף חזק',
    61: 'גשם קל', 63: 'גשם', 65: 'גשם חזק', 80: 'ממטרים', 81: 'ממטרים',
    82: 'ממטרים חזקים', 95: 'סופת רעמים',
}

# נתונים שמורים - עקיפה כיתתית בלבד לגרסת האופליין (לא לעשות את זה בקוד אמיתי)
with open('../../resources/weather_sample.json', encoding='utf-8') as f:
    SAMPLE_WEATHER = json.load(f)
with open('../../resources/quotes_sample.json', encoding='utf-8') as f:
    SAMPLE_QUOTES = json.load(f)

print("הכל מוכן! ✅")

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 1: מזג אוויר</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            כתבו תוכנית שמקבלת שם עיר (מתוך הטבלה <code>CITIES</code>) ומציגה:
        </p>
    </div>
</div>

<ol style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>טמפרטורה נוכחית</li>
    <li>תיאור מזג האוויר הנוכחי (בעזרת <code>WEATHER_CODES</code>)</li>
    <li>תחזית ל-3 הימים הקרובים (מינימום ומקסימום לכל יום)</li>
</ol>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 השתמשו ב-Open-Meteo. קווי הרוחב והאורך של העיר נמצאים ב-<code>CITIES[city]</code><br>
🔹 <strong>קודם הדפיסו את התשובה</strong> (<code>print(data)</code> או <code>json.dumps(data, ensure_ascii=False, indent=2)</code>) ומצאו בה איפה נמצאת הטמפרטורה ואיפה התחזית. זה חלק מהתרגיל<br>
🔹 אחרי שהדפסתם את התשובה תראו ש-<code>data['current']</code> הוא מילון שמכיל את הטמפרטורה (<code>temperature_2m</code>) ואת קוד מזג האוויר (<code>weather_code</code>)<br>
🔹 אחרי שהדפסתם תראו ש-<code>data['daily']</code> מכיל ארבע רשימות באורך 4 (היום ועוד 3 ימים). רשימות מאותו אינדקס שייכות לאותו יום<br>
🔹 את התיאור בעברית מוציאים עם <code>WEATHER_CODES.get(קוד, 'לא ידוע')</code>
</p>

In [ ]:
# פתרו כאן:
city = "חיפה"  # שנו כאן

lat, lon = CITIES[city]
params = {
    'latitude': lat,
    'longitude': lon,
    'current': 'temperature_2m,weather_code',
    'daily': 'weather_code,temperature_2m_max,temperature_2m_min',
    'timezone': 'auto',
    'forecast_days': 4,
}

data = get_api_data(OPEN_METEO_URL, params=params)
if data is None:
    print("⚠️ גרסת אופליין: משתמשים בנתונים שמורים (תל אביב)")
    data = SAMPLE_WEATHER

# קודם הדפיסו את data ומצאו בו את המידע. המשיכו מכאן: הדפיסו טמפרטורה נוכחית, תיאור, ותחזית ל-3 הימים הבאים

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 2: ציטוטים אקראיים</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            כתבו תוכנית שמציגה 5 ציטוטים אקראיים מ-dummyjson.com.<br>
לכל ציטוט הציגו: הטקסט, שם המחבר ואורך הציטוט בתווים.<br>
(שימו לב: ב-API הזה אין תגיות, רק <code>id</code>, <code>quote</code> ו-<code>author</code>.)
        </p>
    </div>
</div>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמז:</strong> הדפיסו קודם את התשובה של בקשה אחת - תראו שיש בה את השדות <code>id</code>, <code>quote</code> ו-<code>author</code>.<br>
אחר כך לולאה של 5 סיבובים; בכל סיבוב קוראים ל-<code>get_api_data(QUOTE_URL)</code>.
בגרסת האופליין: אם קיבלתם <code>None</code>, השתמשו בציטוט מהרשימה <code>SAMPLE_QUOTES</code> (עקיפה כיתתית בלבד).
</p>

In [ ]:
# פתרו כאן (קודם הדפיסו תשובה אחת: print(get_api_data(QUOTE_URL))):
for i in range(5):
    data = get_api_data(QUOTE_URL)
    if data is None:
        data = SAMPLE_QUOTES[i]  # עקיפה כיתתית - גרסת אופליין בלבד

    # הדפיסו: מספר, טקסט, מחבר, אורך (len)

    time.sleep(0.5)  # הפסקה קטנה בין בקשות

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 3: הורדת תמונות</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            כתבו תוכנית שמורידה 10 תמונות של כלבים מ-dog.ceo ושומרת אותן בתיקייה <code>downloads/dogs/</code> עם השמות
<code>dog_1.jpg</code>, <code>dog_2.jpg</code> וכו'.
        </p>
    </div>
</div>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 הכתובת <code>https://dog.ceo/api/breeds/image/random/10</code> מחזירה 10 כתובות בבת אחת. הדפיסו את התשובה ומצאו איפה הרשימה - אחרי שתדפיסו תראו שהיא נמצאת ב-<code>data['message']</code><br>
🔹 צרו את התיקייה עם <code>os.makedirs('downloads/dogs', exist_ok=True)</code><br>
🔹 להורדת כל תמונה: <code>requests.get(כתובת, timeout=10).content</code><br>
🔹 לשמירה: <code>open(נתיב, 'wb')</code> ואז <code>f.write(...)</code><br>
🔹 בסוף הדפיסו כמה קבצים נשמרו עם <code>os.listdir</code>
</p>

In [ ]:
# פתרו כאן:
os.makedirs('downloads/dogs', exist_ok=True)

data = get_api_data('https://dog.ceo/api/breeds/image/random/10')

if data:
    image_urls = data['message']
    # עברו בלולאה על image_urls (enumerate מ-1), הורידו כל תמונה ושמרו אותה

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 4: מעקב שער דולר</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            צרו תוכנית שבודקת את שער הדולר/שקל 5 פעמים, עם הפסקה של 2 שניות בין בדיקה לבדיקה, ושומרת את התוצאות ל-CSV עם העמודות: תאריך, שעה, שער.<br>
השתמשו ב-API של Frankfurter (חינמי, בלי מפתח):<br>
<code>https://api.frankfurter.dev/v1/latest</code> עם הפרמטרים <code>base=USD</code> ו-<code>symbols=ILS</code>.<br>
הערה: השער מתעדכן פעם ביום, לכן כנראה תקבלו חמש פעמים את אותו מספר - המטרה היא לתרגל לולאה, <code>time.sleep</code> וכתיבת CSV.
        </p>
    </div>
</div>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 הדפיסו את התשובה של בקשה אחת ומצאו איפה נמצא השער. אחרי שתדפיסו תראו שהוא נמצא ב-<code>data['rates']['ILS']</code><br>
🔹 שעה נוכחית: <code>datetime.now().strftime('%H:%M:%S')</code><br>
🔹 פתחו את הקובץ במצב <code>'w'</code> (כך אפשר להריץ שוב בלי לצבור שורות ישנות) עם <code>newline=''</code> ו-<code>encoding='utf-8-sig'</code>
</p>

In [ ]:
# פתרו כאן:
FX_URL = 'https://api.frankfurter.dev/v1/latest'
fx_params = {'base': 'USD', 'symbols': 'ILS'}

with open('usd_ils.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['תאריך', 'שעה', 'שער'])

    for i in range(5):
        # 1. שלחו בקשה עם get_api_data(FX_URL, params=fx_params)
        # 2. אם קיבלתם מידע - כתבו שורה עם writer.writerow([...])
        # 3. time.sleep(2)
        pass

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 5: מערכת התראות</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            צרו מערכת שבודקת את מזג האוויר של מחר בעיר לבחירתכם ויוצרת התראה:<br>
אם הטמפרטורה המקסימלית מחר מעל 30 מעלות, או שהמינימלית מתחת ל-10 -
שמרו הודעת התראה לקובץ <code>alerts.txt</code> עם תאריך ושעה.
        </p>
    </div>
</div>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמזים:</strong><br>
🔹 הדפיסו את התשובה ומצאו את התחזית היומית. אחרי שתדפיסו תראו שהיא ב-<code>data['daily']</code>, ושבכל רשימה אינדקס 0 הוא היום ואינדקס 1 הוא מחר<br>
🔹 בנובמבר קשה למצוא יום עם יותר מ-30 מעלות... שנו את הספים בזמן הבדיקה (למשל <code>HOT_LIMIT = 20</code>) כדי לראות שההתראה נכתבת<br>
🔹 מחקו את הקובץ הישן בתחילת התא (<code>os.remove</code>) כדי שאפשר יהיה להריץ שוב, ופתחו את הקובץ במצב <code>'a'</code> (הוספה)
</p>

In [ ]:
# פתרו כאן:
city = "ירושלים"  # שנו כאן
HOT_LIMIT = 30    # שנו כאן לבדיקה
COLD_LIMIT = 10   # שנו כאן לבדיקה

if os.path.exists('alerts.txt'):
    os.remove('alerts.txt')

lat, lon = CITIES[city]
params = {
    'latitude': lat,
    'longitude': lon,
    'daily': 'temperature_2m_max,temperature_2m_min',
    'timezone': 'auto',
    'forecast_days': 2,
}

data = get_api_data(OPEN_METEO_URL, params=params)
if data is None:
    data = SAMPLE_WEATHER

# קודם הדפיסו את data ומצאו בו את התחזית. המשיכו מכאן: שלפו את מחר (אינדקס 1), בדקו את הספים וכתבו ל-alerts.txt

## <p style="text-align: right; direction: rtl; float: right; clear: both;">תרגיל 6 (אתגר): מאגר פוסטים</p>

<div class="align-center" style="display: flex; text-align: right; direction: rtl; clear: both;">
    <div style="display: flex; width: 10%; float: right; clear: both;">
        <img src="images/exercise.svg" style="height: 50px !important;" alt="תרגול">
    </div>
    <div style="width: 90%">
        <p style="text-align: right; direction: rtl; float: right; clear: both;">
        <strong>תרגול</strong>:<br>
            בנו מערכת שאוספת כותרות מ-2 מקורות שמחזירים JSON ושומרת אותן.<br>
(בדקנו: מקורות חדשות אמיתיים דורשים מפתח או קוראים XML, ולכן נשתמש בשני שירותי נתוני בדיקה.)
        </p>
    </div>
</div>

<ul style="text-align: right; direction: rtl; float: right; clear: both;">
    <li>מקור א: <code>https://dummyjson.com/posts?limit=30</code> - הרשימה נמצאת ב-<code>data["posts"]</code>, לכל פוסט יש <code>id</code>, <code>title</code>, <code>body</code></li>
    <li>מקור ב: <code>https://jsonplaceholder.typicode.com/posts?_limit=30</code> - התשובה היא ישר רשימה של פוסטים</li>
    <li>בנו רשימה של מילונים עם: תאריך (של היום), מקור, כותרת, קישור (למשל <code>https://dummyjson.com/posts/{id}</code>)</li>
    <li>שמרו את הרשימה לקובץ <code>posts.json</code></li>
    <li>יצרו דוח Excel <code>posts.xlsx</code> עם openpyxl (כמו ביום 3)</li>
    <li>הוסיפו פילטר לפי מילות מפתח: שמרו רק פוסטים שהכותרת או הגוף שלהם מכילים אחת מהמילים</li>
</ul>

<p style="text-align: right; direction: rtl; float: right; clear: both;">
💡 <strong>רמז:</strong> התחילו ממקור אחד, ורק כשהוא עובד הוסיפו את השני. הדפיסו את התשובה של כל מקור ובדקו אם היא מילון עם רשימה בפנים, או רשימה ישר. לפילטר השתמשו ב-<code>if מילה in טקסט.lower()</code>.
</p>

In [ ]:
# פתרו כאן:
KEYWORDS = ["love", "est"]  # שנו כאן
today = datetime.now().strftime('%Y-%m-%d')
all_posts = []

# מקור א: dummyjson
data_a = get_api_data('https://dummyjson.com/posts', params={'limit': 30})
if data_a:
    for post in data_a['posts']:
        pass  # הוסיפו ל-all_posts מילון עם date, source, title, link

# מקור ב: jsonplaceholder (התשובה היא רשימה)
data_b = get_api_data('https://jsonplaceholder.typicode.com/posts', params={'_limit': 30})
if data_b:
    for post in data_b:
        pass  # הוסיפו ל-all_posts מילון עם date, source, title, link

# המשיכו מכאן: סננו לפי KEYWORDS, שמרו ל-posts.json ול-posts.xlsx

---

<p style="text-align: center; direction: rtl; clear: both;">
    <strong>סיימתם?</strong><br>
    מחר: SQL - ניהול מסדי נתונים!<br>
    <em>בהצלחה! 🐍✨</em>
</p>